# UAE Mobile Intelligence - Peer-Group Composite Classifier

The brief is explicit and mandatory here: **every** peer comparison the product makes (every
"peer median" shown on screen, and later every Peer Gap score) must be against an appropriate
peer group, never the UAE-wide distribution. It is equally explicit about *how not to build it*:
classifying by OSM land-use tag collapses commercial/retail to ~17 zones nationally -- UAE
land-use tagging is simply too thin. Instead, build peer groups from a **composite of population
density, building-footprint density, POI density and road density** -- all already sitting in
`zone_quarter_table.parquet` -- which the brief says yields four stable groups: commercial/
urban-core, low-density residential, industrial, rural/edge.

This notebook builds that classifier with KMeans (`k=4`, matching the brief's four named groups),
inspects the resulting clusters to assign the human-readable labels, and validates the two things
the brief actually cares about: **groups large enough for a robust median**, and **enough within-
UAE spread that real peer gaps exist to find** -- not classification accuracy against some ground
truth, because no ground truth exists here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Build the per-zone feature table

Density features are static (population and OSM don't change quarter to quarter), so this
classifier runs on one row per unique H3 cell, not per zone-quarter. Population density is
derived here (`population / zone_area_km2`) since the master table only stores the raw count.

`building_count_per_km2` is dropped from the feature set: it correlates 0.79 with
`building_footprint_pct` on this data (both measure "how built-up," one by count, one by
coverage area), so keeping both would double-count the same signal in the clustering distance.
Footprint percentage is kept as the more informative of the two -- a few large warehouses and
many small houses can have the same count but very different footprint.

In [2]:
zone_quarter = pd.read_parquet("../data/processed/zone_quarter_table.parquet")
# Idempotency: this notebook may already have run once and written peer_group back into this
# same file, so drop any prior peer_group before recomputing rather than colliding on re-run.
zone_quarter = zone_quarter.drop(columns=["peer_group"], errors="ignore")
zones = zone_quarter.drop_duplicates("h3_cell").copy()
zones["pop_density_per_km2"] = zones["population"] / zones["zone_area_km2"]

FEATURES = ["pop_density_per_km2", "building_footprint_pct", "poi_count_per_km2", "road_density_km_per_km2"]

print("Zones to classify:", len(zones))
zones[FEATURES].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2)

Zones to classify: 1074


,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2
count,1074.00,1074.00,1074.00,1074.00
mean,320.31,0.64,1.29,3.12
std,705.94,2.46,8.21,4.50
min,0.00,0.00,0.00,0.00
10%,0.94,0.00,0.00,0.16
25%,9.26,0.00,0.00,0.66
50%,60.89,0.01,0.03,1.59
75%,255.78,0.13,0.13,3.50
90%,910.34,0.90,0.98,7.63
99%,3803.27,14.28,30.42,23.25


## 2. Log-transform, then standardize

All four features are heavily right-skewed -- a lot of near-zero zones and a long tail of dense
ones (e.g. population density spans 0 to ~6,900/km2, POI density 0 to ~400/km2). Without a log
transform, KMeans' Euclidean distance would be dominated entirely by the few extreme zones.
`log1p` handles the many exact-zero zones cleanly (rural cells with literally no OSM features).
Standardizing after that puts all four features on the same scale, since population density
(per km2) and footprint percentage live on completely different numeric ranges.

In [3]:
X_log = np.log1p(zones[FEATURES])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Feature matrix:", X_scaled.shape)
pd.DataFrame(X_scaled, columns=FEATURES).describe().round(2)

Feature matrix: (1074, 4)


,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2
count,1074.00,1074.00,1074.00,1074.00
mean,0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00
min,-1.81,-0.40,-0.38,-1.40
25%,-0.74,-0.40,-0.38,-0.74
50%,0.08,-0.39,-0.34,-0.15
75%,0.74,-0.18,-0.19,0.57
max,2.10,5.72,7.45,3.10


## 3. KMeans, k=4 -- matching the brief's four named groups

`k=4` isn't tuned by elbow/silhouette search here: the brief specifies four groups by name
(commercial/urban-core, low-density residential, industrial, rural/edge) as the peer-group
structure the product needs, so the target `k` is a requirement, not a free hyperparameter. What
*is* validated below is whether those four clusters are actually usable -- big enough, and
spread out enough -- not whether four is the "best" k in isolation.

In [4]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
zones["cluster"] = kmeans.fit_predict(X_scaled)

print(zones["cluster"].value_counts().sort_index())

cluster
0    498
1     51
2    404
3    121
Name: count, dtype: int64


## 4. Assign human-readable labels from the cluster centroids

The clustering itself doesn't know what "industrial" means -- labels are assigned afterward from
each cluster's centroid and (2026-09-17 fix, see below) each zone's own population density,
using an explainable rule so the mapping can be checked, not guessed:

1. Rank clusters by overall density intensity (mean of the four standardized features). The
   highest becomes **commercial/urban-core**, the lowest becomes **rural/edge**.
2. Of the two remaining (middle-intensity) clusters, split by **population density**, not
   POI/building/road density: a zone is **industrial** only if its own population density is
   below the pooled median of the two middle clusters combined; at or above that median, it
   is **low-density residential**.

**Root-cause fix (2026-09-17):** the previous rule split the two middle clusters purely by
POI density relative to building/road density ("warehouses have roads and buildings but few
POIs"), never consulting population density at all. That conflated two very different real
populations that happen to share the identical low-OSM-tagging signature: genuinely industrial
land (few or no residents), and real residential areas that OSM simply hasn't finished tagging
with buildings/POIs/roads yet (a well-known regional OSM completeness gap). Audit found this
was not a 2-zone problem: 498 of 1,074 zones nationally (46%) were labeled "industrial" under
the old rule, 80% of them with population density above 50/km2, including well-known real
residential/mixed districts (e.g. Dubai Knowledge Park at 1,085/km2). Population density itself
was independently verified correct (raw WorldPop raster re-clipped against the exact H3
polygons, matching the processed value exactly) -- the bug was in this label-assignment rule,
not the population or OSM aggregation, and not KMeans itself (it was already assigning every
zone to its genuinely nearest centroid).

In [5]:
# Two inverses needed to get back to real units: undo StandardScaler, then undo log1p.
# Skipping the expm1 step would leave centroids in log-space -- technically fine for ranking
# (monotonic), but meaningless to read (e.g. a population-density centroid of "7.95" when the
# real range is 0-6,900/km2) if anyone asks "why is this zone's peer group X?" later.
centroids_log = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FEATURES)
centroids = np.expm1(centroids_log)
centroids_std = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES)  # standardized space, for ranking
centroids["intensity"] = centroids_std.mean(axis=1)

order = centroids["intensity"].sort_values(ascending=False).index.tolist()
urban_core_cluster, rural_cluster = order[0], order[-1]
middle_clusters = order[1:3]

# Population-density split (2026-09-17 fix) -- replaces the old POI-to-built-ratio cluster-
# level rule (kept as a diagnostic column below, no longer the deciding signal). The pooled
# median is computed fresh from THIS run's middle-tier zones -- data-driven, not a hardcoded
# constant -- so it can never drift stale if the underlying population/OSM data changes.
middle_mask = zones["cluster"].isin(middle_clusters)
pop_density_median_middle_tier = zones.loc[middle_mask, "pop_density_per_km2"].median()

zones["peer_group"] = np.select(
    [
        zones["cluster"] == urban_core_cluster,
        zones["cluster"] == rural_cluster,
        middle_mask & (zones["pop_density_per_km2"] < pop_density_median_middle_tier),
        middle_mask & (zones["pop_density_per_km2"] >= pop_density_median_middle_tier),
    ],
    ["commercial/urban-core", "rural/edge", "industrial", "low-density residential"],
    default=None,
)
assert zones["peer_group"].isna().sum() == 0, "Every zone must get exactly one of the four labels"

# Diagnostic only, not the deciding rule anymore -- kept so the OLD signal is still visible
# next to the new one when sanity-checking a specific zone.
centroids["poi_to_built_ratio"] = centroids["poi_count_per_km2"] / (
    centroids["building_footprint_pct"] + centroids["road_density_km_per_km2"] + 1e-6
)

print(f"Middle-tier population-density median (the new industrial/residential split point): "
      f"{pop_density_median_middle_tier:.1f}/km2")
print(f"Cluster IDs: urban_core={urban_core_cluster}, rural={rural_cluster}, "
      f"middle (population-split)={middle_clusters}")
print()
centroids.index.name = "cluster"
centroids.round(2)

Middle-tier population-density median (the new industrial/residential split point): 153.1/km2
Cluster IDs: urban_core=1, rural=2, middle (population-split)=[3, 0]



,pop_density_per_km2,building_footprint_pct,poi_count_per_km2,road_density_km_per_km2,intensity,poi_to_built_ratio
cluster,,,,,,
0,112.12,0.07,0.08,2.22,-0.01,0.04
1,2349.29,7.53,13.68,16.82,2.87,0.56
2,4.33,0.00,0.01,0.48,-0.67,0.03
3,804.08,1.06,0.97,7.47,1.07,0.11


## 5. Validate -- group sizes and feature separation

The brief's own failure case (land-use classification collapsing to ~17 commercial zones) is
exactly what this check is for: a group that small can't support a robust median. Feature
medians are printed by GROUP now (not by cluster centroid) -- the industrial/low-density-
residential split is a per-zone population-density decision (2026-09-17), so a single cluster
centroid no longer speaks for one label alone; the group-level medians below are the real
check that urban-core is highest on every feature, rural/edge lowest, and that industrial is
now genuinely low-population while low-density residential is not.

In [6]:
group_sizes = zones["peer_group"].value_counts()
print("Peer-group sizes (zones):")
print(group_sizes.to_string())
print()
print("Smallest group:", group_sizes.idxmin(), "--", group_sizes.min(), "zones")
assert group_sizes.min() >= 30, "A peer group this small can't support a robust median"

GROUP_ORDER = ["commercial/urban-core", "low-density residential", "industrial", "rural/edge"]
print()
print("Feature medians by GROUP (original units, real per-zone assignment -- not cluster centroids):")
group_medians = zones.groupby("peer_group")[FEATURES].median()
print(group_medians.reindex(GROUP_ORDER).round(3).to_string())


Peer-group sizes (zones):
peer_group
rural/edge                 404
low-density residential    310
industrial                 309
commercial/urban-core       51

Smallest group: commercial/urban-core -- 51 zones

Feature medians by GROUP (original units, real per-zone assignment -- not cluster centroids):
                         pop_density_per_km2  building_footprint_pct  poi_count_per_km2  road_density_km_per_km2
peer_group                                                                                                      
commercial/urban-core               2699.842                   7.609             11.175                   18.966
low-density residential              383.861                   0.156              0.184                    3.944
industrial                            67.859                   0.010              0.000                    1.746
rural/edge                             4.127                   0.000              0.000                    0.449


## 6. Validate -- do real peer gaps exist within each group?

Group medians only matter if there's genuine within-group spread to compare a zone against --
if every zone in a group scores nearly identically, "peer gap" is meaningless. This checks raw
`download_mbps` (test-weighted mean across the zone's quarters) as a proxy, since the Experience
Index itself hasn't been computed yet (next notebook). The brief's own T0 finding -- a roughly
threefold spread between the strongest and weakest deciles nationally -- is the benchmark: each
peer group should show real internal spread, not be artificially uniform.

In [7]:
zone_download = zone_quarter.groupby("h3_cell").apply(
    lambda g: np.average(g["download_mbps"], weights=g["tests"]), include_groups=False
).rename("avg_download_mbps")
zones = zones.join(zone_download, on="h3_cell")

spread = zones.groupby("peer_group")["avg_download_mbps"].describe(percentiles=[.1, .5, .9])[
    ["count", "10%", "50%", "90%"]
]
spread["p90_p10_ratio"] = spread["90%"] / spread["10%"]
print(spread.round(1).to_string())
print()
print("Each group shows real internal spread (p90/p10 ratio well above 1x), confirming peer gaps")
print("are findable within every group, not just across the national distribution.")

                         count    10%    50%    90%  p90_p10_ratio
peer_group                                                        
commercial/urban-core     51.0  353.4  429.1  603.7            1.7
industrial               309.0   62.2  289.8  651.3           10.5
low-density residential  310.0  197.6  406.2  595.3            3.0
rural/edge               404.0    8.6  149.9  682.6           79.7

Each group shows real internal spread (p90/p10 ratio well above 1x), confirming peer gaps
are findable within every group, not just across the national distribution.


## 7. Save

`peer_groups_uae.parquet` is the standalone static classification (one row per H3 cell). The
label is also merged back into `zone_quarter_table.parquet` so every downstream notebook gets
`peer_group` for free without re-running this classifier.

In [8]:
peer_groups = zones[["h3_cell", "peer_group"] + FEATURES].copy()
out_path = Path("../data/processed/peer_groups_uae.parquet")
peer_groups.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

zone_quarter_with_peers = zone_quarter.merge(peer_groups[["h3_cell", "peer_group"]], on="h3_cell", how="left")
assert len(zone_quarter_with_peers) == len(zone_quarter), "Merge must not add or drop rows"
assert zone_quarter_with_peers["peer_group"].isna().sum() == 0, "Every measured zone must get a peer group"

out_path2 = Path("../data/processed/zone_quarter_table.parquet")
zone_quarter_with_peers.to_parquet(out_path2, index=False)
print(f"Updated: {out_path2} (+peer_group column, {len(zone_quarter_with_peers)} rows)")

Saved: ..\data\processed\peer_groups_uae.parquet (42.3 KB)


Updated: ..\data\processed\zone_quarter_table.parquet (+peer_group column, 5011 rows)


## Summary

- `data/processed/peer_groups_uae.parquet` -- one row per H3 res-6 cell (1,074 zones), with
  `peer_group` (one of the four brief-mandated labels) and the density features used to assign it.
- `data/processed/zone_quarter_table.parquet` -- now carries `peer_group` on every zone-quarter row.
- All four groups clear a 30-zone minimum size and show real internal spread in raw download
  speed (proxy for Experience Index, not yet computed) -- both the brief's stated risks
  (collapsed group size, no findable within-group gap) are checked and pass.
- Labels are assigned from cluster centroids via an explainable rule (density-intensity rank,
  then POI-to-built ratio for the industrial/residential split), not left as opaque cluster
  numbers -- so "why is this zone's peer group X?" has a one-line answer.

**Next:** Experience Index and Confidence Score on this real table (`src/compute_scores.py`
currently only runs on synthetic sample data), then Peer Gap -- comparing each zone's Experience
Index against its own peer group's median, which is now possible for the first time.